# DBSCAN na prática: regiões de ocorrências
Envie o arquivo `crimes.csv` quando a célula 2 pedir. Os dados são inventados, só para a aula.

In [ ]:
# 1. Importar bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from google.colab import files

In [ ]:
# 2. Enviar o arquivo crimes.csv
files.upload()
df = pd.read_csv("crimes.csv")
df.head()

In [ ]:
# 3. Olhar as ocorrências no mapa (ainda sem grupos)
plt.scatter(df["x_km"], df["y_km"], s=20, color="gray")
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("Ocorrências registradas")
plt.show()

In [ ]:
# 4. Padronizar as coordenadas (o eps é uma distância)
X = StandardScaler().fit_transform(df[["x_km", "y_km"]])

In [ ]:
# 5. Escolher o eps: gráfico de k-distâncias
# 2 colunas -> min_samples = 4, então olhamos o 4º vizinho mais próximo
vizinhos = NearestNeighbors(n_neighbors=4).fit(X)
distancias, _ = vizinhos.kneighbors(X)
k_dist = sorted(distancias[:, -1])

plt.plot(k_dist)
plt.xlabel("Pontos ordenados")
plt.ylabel("Distância ao 4º vizinho")
plt.title("Gráfico de k-distâncias")
plt.show()

In [ ]:
# 6. Aplicar o DBSCAN
modelo = DBSCAN(eps=0.25, min_samples=4)
df["dbscan"] = modelo.fit_predict(X)

# -1 = ruído (não entrou em nenhum grupo)
print(df["dbscan"].value_counts().sort_index())

In [ ]:
# 7. Visualizar os grupos (ruído em cinza)
ruido = df["dbscan"] == -1

plt.scatter(df.loc[~ruido, "x_km"], df.loc[~ruido, "y_km"],
            c=df.loc[~ruido, "dbscan"], cmap="viridis", s=25)
plt.scatter(df.loc[ruido, "x_km"], df.loc[ruido, "y_km"],
            c="lightgray", marker="x", s=25, label="ruído")
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("Regiões encontradas pelo DBSCAN")
plt.legend()
plt.show()

In [ ]:
# 8. Testar outros valores de eps
for eps in [0.05, 0.1, 0.25, 0.6]:
    rotulos = DBSCAN(eps=eps, min_samples=4).fit_predict(X)
    grupos = len(set(rotulos)) - (1 if -1 in rotulos else 0)
    print(f"eps={eps}: {grupos} grupos, {(rotulos == -1).sum()} pontos de ruído")

In [ ]:
# 9. Interpretar: o que tem em cada região?
regioes = df[df["dbscan"] != -1]
regioes.groupby("dbscan").agg(
    ocorrencias=("ocorrencia_id", "count"),
    hora_media=("hora", "mean"),
    tipo_mais_comum=("tipo", lambda t: t.mode()[0])).round(1)

In [ ]:
# 10. Comparar com o K-Means (K = 3)
df["kmeans"] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X)

fig, eixos = plt.subplots(1, 2, figsize=(11, 4))
eixos[0].scatter(df["x_km"], df["y_km"], c=df["kmeans"], cmap="viridis", s=25)
eixos[0].set_title("K-Means (K = 3)")
eixos[1].scatter(df.loc[~ruido, "x_km"], df.loc[~ruido, "y_km"],
                 c=df.loc[~ruido, "dbscan"], cmap="viridis", s=25)
eixos[1].scatter(df.loc[ruido, "x_km"], df.loc[ruido, "y_km"],
                 c="lightgray", marker="x", s=25)
eixos[1].set_title("DBSCAN")
plt.show()

# Quem caiu em cada grupo do K-Means?
pd.crosstab(df["dbscan"], df["kmeans"])